In [245]:
import random
from collections import defaultdict
import numpy as np

This is a function to track stats for a cube night with 7 people. I make the following assumptions:
1. The length a match of Magic takes is normally distributed (mean = 40 minutes; standard deviation = 8 minutes). You can change this if you want (look at the 2nd line of the code).
2. Matches continue until everyone has played at least 3 matches. The remaining matches conclude and give us the final match time. This means some players are playing more than 3 matches.
3. Players will never play a match against someone they have previously played against. I don't have a completely systematic way to ensure this; therefore, I throw an error if the matchups end up sufficiently 'unresolvable'. If someone has interest, I can explain exactly what I mean by this. However, it only happens about 1/1000 times, so it shouldn't affect our results enough for us to care.

In [406]:
def round_update(matches_played, x1, x2):
    round_time = random.gauss(mu=40, sigma=8) # Edit this if you want to change the Magic match-length distribution
    matches_played[x1].append(x2)
    matches_played[x2].append(x1)
    return matches_played, round_time
def print_info(ongoing_rounds, current_pairings, remaining_players):
    print(f"Ongoing round times: {ongoing_rounds}")
    print(f"Current round pairings: {current_pairings}")
    print(f"Currently unpaired: {remaining_players}")
    print("")
    return
def append_info(round_times, ongoing_rounds, pairings, current_pairings, p1, p2, new_time):
    round_times.append(new_time)
    ongoing_rounds.append(new_time)
    pairings.append([p1,p2])
    current_pairings.append([p1,p2])
    return round_times, ongoing_rounds, pairings, current_pairings
def count_matches_played(matches_played):
    sum_matches = 0
    for matches in matches_played.values():
        sum_matches += len(matches)
    matches_played = sum_matches/2
    return matches_played
def cube_w_7():
    matches_played = defaultdict(list)
    round_times = []
    ongoing_rounds = []
    pairings = []
    current_pairings = []
    players = [1,2,3,4,5,6,7]
    
    random.shuffle(players)
    remaining_players = players.copy()
    
    for match in range(3):
        p1 = remaining_players.pop()
        p2 = remaining_players.pop()
        matches_played, round_time = round_update(matches_played, p1, p2)
        round_times.append(round_time)
        ongoing_rounds.append(round_time)
        pairings.append([p1,p2])
        current_pairings.append([p1,p2])
    #print_info(ongoing_rounds, current_pairings, remaining_players)
    
    fewest_matches = 0
    while fewest_matches < 3:
        time_to_finish = min(ongoing_rounds)
        next_to_finish = ongoing_rounds.index(time_to_finish)
        p1 = remaining_players.pop()
        
        if current_pairings[next_to_finish][0] not in matches_played[p1]:
            p2 = current_pairings[next_to_finish][0]
            p2_finish_time = time_to_finish
            remaining_players.append(current_pairings[next_to_finish][1])
            matches_played, round_time = round_update(matches_played, p1, p2)
            ongoing_rounds.pop(next_to_finish)
            current_pairings.pop(next_to_finish)
            new_time = round_time + time_to_finish
            round_times, ongoing_rounds, pairings, current_pairings = append_info(round_times, ongoing_rounds, pairings, current_pairings, p1, p2, new_time)
            #print_info(ongoing_rounds, current_pairings, remaining_players)
        elif current_pairings[next_to_finish][1] not in matches_played[p1]:
            p2 = current_pairings[next_to_finish][1]
            remaining_players.append(current_pairings[next_to_finish][0])
            matches_played, round_time = round_update(matches_played, p1, p2)
            ongoing_rounds.pop(next_to_finish)
            current_pairings.pop(next_to_finish)
            new_time = round_time + time_to_finish
            round_times, ongoing_rounds, pairings, current_pairings = append_info(round_times, ongoing_rounds, pairings, current_pairings, p1, p2, new_time)
            #print_info(ongoing_rounds, current_pairings, remaining_players)
        else:
            waiting_ppl = current_pairings.pop(next_to_finish)
            waiting_round_end = ongoing_rounds.pop(next_to_finish)
            remaining_players.extend(waiting_ppl)
            
            time_to_finish = min(ongoing_rounds)
            next_to_finish = ongoing_rounds.index(time_to_finish)
            if current_pairings[next_to_finish][0] not in matches_played[p1]:
                p2 = current_pairings[next_to_finish][0]
                remaining_players.append(current_pairings[next_to_finish][1])
                matches_played, round_time = round_update(matches_played, p1, p2)
                ongoing_rounds.pop(next_to_finish)
                current_pairings.pop(next_to_finish)
                new_time = round_time + time_to_finish
                round_times, ongoing_rounds, pairings, current_pairings = append_info(round_times, ongoing_rounds, pairings, current_pairings, p1, p2, new_time)
                #print_info(ongoing_rounds, current_pairings, remaining_players)        
            elif current_pairings[next_to_finish][1] not in matches_played[p1]:
                p2 = current_pairings[next_to_finish][1]
                remaining_players.append(current_pairings[next_to_finish][0])
                matches_played, round_time = round_update(matches_played, p1, p2)
                ongoing_rounds.pop(next_to_finish)
                current_pairings.pop(next_to_finish)
                new_time = round_time + time_to_finish
                round_times, ongoing_rounds, pairings, current_pairings = append_info(round_times, ongoing_rounds, pairings, current_pairings, p1, p2, new_time)
                #print_info(ongoing_rounds, current_pairings, remaining_players)
            else:
                if fewest_matches >= 3:
                    final_time = max(round_times)
                    number_of_matches = count_matches_played(matches_played)
                    return final_time, number_of_matches
                raise TypeError("Unable to create new matchup 1")
    
            p1 = remaining_players.pop()
            if remaining_players[0] not in matches_played[p1]:
                p2 = remaining_players.pop(0)
                matches_played, round_time = round_update(matches_played, p1, p2)
                new_time = round_time + waiting_round_end
                round_times, ongoing_rounds, pairings, current_pairings = append_info(round_times, ongoing_rounds, pairings, current_pairings, p1, p2, new_time)
                #print_info(ongoing_rounds, current_pairings, remaining_players)        
            elif remaining_players[1] not in matches_played[p1]:
                p2 = remaining_players.pop(1)
                matches_played, round_time = round_update(matches_played, p1, p2)
                new_time = round_time + waiting_round_end
                round_times, ongoing_rounds, pairings, current_pairings = append_info(round_times, ongoing_rounds, pairings, current_pairings, p1, p2, new_time)
                #print_info(ongoing_rounds, current_pairings, remaining_players)
            else:
                if fewest_matches >= 3:
                    final_time = max(round_times)
                    return final_time, number_of_matches                
                raise TypeError("Unable to create new matchup 2")
    
        player_w_least_matches = min(matches_played.values(), key=len)
        fewest_matches = len(player_w_least_matches)
        
    #print(f"Matches played: {dict(matches_played)}")
    #print(f"Round times: {round_times}")
    #print(f"All round pairings: {pairings}")
    final_time = max(round_times)
    number_of_matches = count_matches_played(matches_played)
    return final_time, number_of_matches

Here is a simulation of 10,000 cube nights with 7 players. I added succussful trials to show that the failure rate is in-fact <1/1000. 

Since people are playing more than 3 matches on average, we account for this by looking at how many matches (on average) a person can complete in 150 minutes. For the set of all game nights $A$, the calculation is:
$$\sum_{i \in A}\frac{2*(\text{number of matches})_i}{(\text{number of players})}*\frac{(150\text{ min})}{(\text{final match time})_i} = \frac{300}{7}\sum_{i \in A}\frac{(\text{number of matches})_i}{(\text{final match time})_i}$$
The multiplication by 2 is because each match consists of 2 people.

In [407]:
final_times_7 = []
number_matches_7 = []
wait_times_7 = []
for _ in range(10000):
    try:
        final_time, number_of_matches = cube_w_7()
        wait_times_7.append(number_of_matches/final_time)
        final_times_7.append(final_time)
        number_matches_7.append(number_of_matches)
    except:
        pass
print(f"Successfull trials: {len(final_times_7)}")
print(f"Average time for the last match to finish: {np.mean(final_times_7)}")
print(f"Average number of matches completed per-person in 150 minutes: {np.mean(wait_times_7)*300/7}")

Successfull trials: 9999
Average time for the last match to finish: 167.46682928465646
Average number of matches completed per-person in 150 minutes: 2.956569755564737


Here is the code for the 8 player cube night, for comparison.

In [408]:
def cube_w_8(mu = 40, sigma = 8):
    final_time = 0
    waiting_time = 0
    for _ in range(3):
        r1 = random.gauss(mu=mu, sigma=sigma)
        r2 = random.gauss(mu=mu, sigma=sigma)
        r3 = random.gauss(mu=mu, sigma=sigma)
        round_time = max([r1, r2, r3])
        final_time += round_time
        waiting_time += 2*(round_time - r1) + 2*(round_time - r2) + 2*(round_time - r3)
    return final_time, waiting_time

The calculation for averagte matches per-person in 150 minutes is simpler here (since there are always 12 matches played):
$$\frac{150*12*2}{8*(\text{avg final time})} $$


In [409]:
final_times_8 = []
waiting_times_8 = []
for _ in range(10000):
    final_time, waiting_time = cube_w_8()
    final_times_8.append(final_time)
    waiting_times_8.append(waiting_time)
mean_final = np.mean(final_times_8)
print(f"Average time for the last match to finish: {mean_final}")
print(f"Average number of matches completed per-person in 150 minutes: {2*12*150/mean_final/8}")

Average time for the last match to finish: 140.38590309838563
Average number of matches completed per-person in 150 minutes: 3.205450049244829


In conclusion, it seems like matches are being more efficiently played with 8 people, even with all the added waiting times. I played around with the distribution choices a little, but it didn't seem to change that result.